# Graph-Based Fraud Ring Detection Walkthrough

This notebook demonstrates end-to-end detection of synthetic financial fraud rings using:
- Synthetic data simulation with realistic topological confounders
- Directed transaction graph and device sharing footprint modeling
- Node and community topological feature engineering (cycles, reciprocity, Louvain)
- Unsupervised heuristic community scoring
- Supervised HistGradientBoosting with GroupKFold cross-validation by community
- Performance evaluation and network visualization

In [ ]:
import sys
from pathlib import Path

# Ensure project root and src directory are on sys.path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

## 1. Simulate Synthetic Transactions & Fraud Rings

We generate 3,000 accounts and 12 embedded fraud rings, along with realistic confounders such as family device sharing and peripheral ring members.

In [ ]:
from fraud_rings.simulate import simulate

accounts_df, transfers_df = simulate(n_accounts=3000, n_rings=12, seed=5)
print(f"Accounts: {len(accounts_df)}, Transfers: {len(transfers_df)}")
print(f"Ring member share: {accounts_df['is_ring_member'].mean():.4f}")
accounts_df.head()

## 2. Construct Transaction & Device Graphs

In [ ]:
from fraud_rings.graph import build_device_graph, build_graph

G = build_graph(accounts_df, transfers_df)
device_G = build_device_graph(accounts_df)

print(f"Directed graph nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
print(f"Device graph edges: {device_G.number_of_edges()}")

## 3. Extract Node and Community Features

Engineered signals include in/out degrees, weighted amounts, PageRank, clustering, k-core number, bounded cycle counts (lengths 2-4), reciprocity, account age, shared device count, amount similarity (CV), and Louvain community size and density.

In [ ]:
from fraud_rings.features import node_features

features_df = node_features(G, device_G, accounts_df)
print(f"Feature matrix shape: {features_df.shape}")
features_df.head()

## 4. Unsupervised Heuristic Detection

Rank Louvain communities by combining density, cycle ratio, mean account age (younger = riskier), device sharing, and amount similarity.

In [ ]:
from fraud_rings.detect import detect_rings_unsupervised

heuristic_comms = detect_rings_unsupervised(features_df, G)
heuristic_comms.head(10)

## 5. Supervised Detection (HistGradientBoosting with GroupKFold)

Partition by Louvain community to ensure no cross-community data leakage, yielding clean out-of-fold probability estimates.

In [ ]:
from fraud_rings.detect import detect_rings_supervised

node_preds_df, sup_comms_df = detect_rings_supervised(features_df, accounts_df, seed=5)
node_preds_df.head(10)

## 6. Evaluation Metrics

We evaluate node-level classification (ROC-AUC, PR-AUC, Precision@k) and community-level retrieval (Precision@10, Ring Recall).

In [ ]:
from fraud_rings.evaluate import evaluate_all

metrics = evaluate_all(node_preds_df, sup_comms_df, accounts_df)
for metric_name, val in metrics.items():
    print(f"{metric_name:30s}: {val:.4f}")

## 7. Visualize the Top Detected Fraud Ring

Plot the top-ranked community and its 1-hop neighborhood.

In [ ]:
from fraud_rings.visualize import plot_top_ring

top_comm_members = sup_comms_df.iloc[0]["member_ids"]
plot_top_ring(
    G=G,
    accounts=accounts_df,
    community_nodes=top_comm_members,
    path="../artifacts/top_ring.png",
)
print("Saved visualization to ../artifacts/top_ring.png")